# Лабораторная работа № 1
## Разведочный анализ больших данных с использованием фреймворка Apache Spark
### Часть 2. Разведочный анализ и очистка данных

**Цель части:** найти в данных дубликаты, ошибки и пропуски, исправить их, описать распределения и связи признаков, сохранить очищенную таблицу для лабораторной № 2.

**Вход:** таблица `sobd_accidents.us_accidents_raw` из части 1. **Выход:** таблица `sobd_accidents.us_accidents_clean`.

### Оглавление

1. [Подготовка](#ch1)
2. [Типы признаков](#ch2)
3. [Дубликаты](#ch3)
4. [Лишние столбцы](#ch4)
5. [Ошибочные значения](#ch5)
6. [Пропуски](#ch6)
7. [Новые признаки](#ch7)
8. [Сохранение очищенной таблицы](#ch8)
9. [Статистические показатели](#ch9)
10. [Распределения](#ch10)
11. [Тяжесть ДТП и источники данных](#ch11)
12. [Корреляции](#ch12)
13. [Выводы](#ch13)

<a id="ch1"></a>
## 1. Подготовка

**Путь:** подключаем библиотеки, запускаем Spark-сессию и открываем таблицу из части 1.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

user = os.getenv("USER")

# Все пути — от папки проекта SOBD-labs: ноутбук лежит в SOBD/2024/lab1 или lab2,
# а в HDFS проект занимает папку SOBD-labs домашнего каталога (data/, warehouse/, models/)
BASE = Path.cwd().parent
HDFS_ROOT = f"hdfs:///user/{user}/SOBD-labs"

# Ресурсы. Узел кафедры (256 ядер, 188 ГБ) делят много студентов — берём ~12 %;
# на локальном стенде (4 ядра, 8 ГБ) — то, что помещается в его YARN
on_cluster = os.cpu_count() >= 64
executors, cores, memory, driver_memory = (4, 8, "6g", "4g") if on_cluster else (2, 2, "1536m", "2g")
# библиотека Iceberg: на кластере скачивается из Maven, на стенде уже лежит среди jar-файлов Spark
iceberg_package = "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.0" if on_cluster else ""

spark = (
    SparkSession.builder
    .appName("lab1-part2-eda")
    .master("yarn")
    .config("spark.dynamicAllocation.enabled", "false")  # ровно столько исполнителей, сколько задали
    .config("spark.executor.instances", executors)
    .config("spark.executor.cores", cores)
    .config("spark.executor.memory", memory)
    .config("spark.driver.memory", driver_memory)
    .config("spark.sql.shuffle.partitions", 4 * executors * cores)  # 4 раздела на ядро вместо 200
    # Iceberg: таблицы и их метаданные — в HDFS
    .config("spark.jars.packages", iceberg_package)
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.spark_catalog", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.spark_catalog.type", "hadoop")
    .config("spark.sql.catalog.spark_catalog.warehouse", f"{HDFS_ROOT}/warehouse")
    # без кеша каталога: после перезаписи таблицы сессия сразу видит новую версию
    .config("spark.sql.catalog.spark_catalog.cache-enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
spark.catalog.setCurrentDatabase("sobd_accidents")

sns.set_theme(style="whitegrid")
FIGURES = BASE / "figures"  # SOBD/2024/figures
FIGURES.mkdir(exist_ok=True)


def save(name):
    """Сохраняет текущий рисунок для главы курсовой."""
    plt.savefig(f"{FIGURES}/{name}.png", dpi=150, bbox_inches="tight")

In [ ]:
df = spark.table("us_accidents_raw")
rows_raw = df.count()
print(f"Строк: {rows_raw:,}, столбцов: {len(df.columns)}")

<a id="ch2"></a>
## 2. Типы признаков

**Путь:** смотрим типы столбцов, а у строковых — число разных значений. По нему видно, где категория, а где текст или идентификатор.

In [ ]:
pd.Series(dict(df.dtypes)).value_counts()

In [ ]:
string_cols = [c for c, t in df.dtypes if t == "string"]

(df.select([F.approx_count_distinct(c).alias(c) for c in string_cols])
   .toPandas().T
   .rename(columns={0: "уникальных значений"})
   .sort_values("уникальных значений"))

По числу значений признаки делятся так:

| Тип | Признаки |
| --- | --- |
| Количественные | `start_lat`, `start_lng`, `end_lat`, `end_lng`, `distance_mi`, `temperature_f`, `wind_chill_f`, `humidity_pct`, `pressure_in`, `visibility_mi`, `wind_speed_mph`, `precipitation_in` |
| Порядковый | `severity` — тяжесть от 1 до 4 |
| Категориальные, до полутора сотен значений | `country`, `source`, `timezone`, `sunrise_sunset` и три вида сумерек, `wind_direction`, `state`, `weather_condition` |
| Категориальные, тысячи и больше значений | `county`, `airport_code`, `city`, `street`, `zipcode` |
| Бинарные | 13 признаков «объект рядом»: `amenity` … `turning_loop` |
| Временные | `start_time`, `end_time`, `weather_timestamp` |
| Текст и идентификатор | `description`, `id` — значений почти столько же, сколько строк |

<a id="ch3"></a>
## 3. Дубликаты

**Путь:** оставляем по одной копии записей, совпадающих во всём, кроме `id`, → считаем, сколько было лишних.

Дальше по таблице будет много проходов, поэтому результат сразу кешируем: Spark не будет заново искать дубликаты при каждом действии.

In [ ]:
cols_without_id = [c for c in df.columns if c != "id"]

df = df.dropDuplicates(cols_without_id).cache()
unique_rows = df.count()

print(f"Строк после удаления дубликатов: {unique_rows:,}")
print(f"Удалено дубликатов: {rows_raw - unique_rows:,} ({(rows_raw - unique_rows) / rows_raw:.2%})")

Полтора процента записей были повторами одного и того же ДТП под разными `id`.

<a id="ch4"></a>
## 4. Лишние столбцы

**Путь:** ищем столбцы, которые не несут информации или повторяют другие, и удаляем их.

Сначала — столбцы с одним значением.

In [ ]:
df.select(F.countDistinct("country").alias("country"),
          F.countDistinct("turning_loop").alias("turning_loop")).toPandas()

Страна всегда `US`, разворотной петли нет ни у одного ДТП: это константы, удаляем.

Четыре столбца времени суток отличаются только определением сумерек. Проверим, как часто они совпадают с основным `sunrise_sunset`.

In [ ]:
twilight = ["civil_twilight", "nautical_twilight", "astronomical_twilight"]

df.select([F.avg((F.col(c) == F.col("sunrise_sunset")).cast("int")).alias(c) for c in twilight]).toPandas().round(3)

Совпадают в 86–95 % случаев — это почти копии. Оставляем один `sunrise_sunset`.

Координаты конца участка `end_lat`, `end_lng` пусты почти у половины записей. Проверим, у каких источников.

In [ ]:
(df.groupBy("source")
   .agg(F.count("*").alias("records"),
        F.avg(F.col("end_lat").isNull().cast("int")).alias("end_lat_null"))
   .orderBy("source")
   .toPandas())

`Source2` и `Source3` конец участка не передают вовсе, а его длина уже есть в `distance_mi`. Удаляем координаты конца.

Удаляем также:
* `description` — свободный текст, почти у каждой записи свой;
* `zipcode`, `airport_code` — сотни тысяч и тысячи значений, место уже задано штатом, городом и координатами;
* `weather_timestamp` — служебное время замера погоды.

In [ ]:
drop_cols = ["country", "turning_loop", *twilight, "end_lat", "end_lng",
             "description", "zipcode", "airport_code", "weather_timestamp"]

df = df.drop(*drop_cols)
print(f"Удалено столбцов: {len(drop_cols)}, осталось: {len(df.columns)}")

<a id="ch5"></a>
## 5. Ошибочные значения

**Путь:** смотрим крайние значения количественных признаков → сверяем с физическим смыслом → ошибки превращаем в пропуски (их заполним в главе 6), выбросы обрезаем.

Добавляем длительность влияния ДТП на движение: через неё проверяется согласованность времени начала и конца.

In [ ]:
df = df.withColumn("duration_min", (F.unix_timestamp("end_time") - F.unix_timestamp("start_time")) / 60)

num_cols = ["distance_mi", "duration_min", "temperature_f", "humidity_pct",
            "pressure_in", "visibility_mi", "wind_speed_mph"]

(df.select(num_cols)
   .summary("min", "1%", "50%", "99%", "99.9%", "max")
   .toPandas().set_index("summary").astype(float).T.round(2))

Ящики с усами по 1 %-й выборке показывают, насколько далеко хвосты уходят от основной массы.

In [ ]:
sample = df.select(num_cols).sample(0.01, seed=42).toPandas()

fig, axes = plt.subplots(1, len(num_cols), figsize=(16, 4))
for ax, c in zip(axes, num_cols):
    sns.boxplot(y=sample[c], ax=ax)
    ax.set_title(c)
    ax.set_ylabel("")
plt.tight_layout()
save("boxplots_raw")
plt.show()

Что нашли и что с этим делаем:

| Признак | Найдено | Почему ошибка | Действие |
| --- | --- | --- | --- |
| `temperature_f` | −89 °F и 207 °F | таких температур в местах ДТП не бывает; −35 °F записано даже во Флориде | вне (−35; 120] → `NULL` |
| `wind_speed_mph` | 1087 миль/ч | ураганный ветер (≥ 55) при ясной погоде невозможен | ≥ 55 → `NULL` |
| `pressure_in` | 0 и 58,6 | давление у земли держится в пределах 20–32 дюймов | вне [20; 32] → `NULL` |
| `visibility_mi` | 140 миль | метеостанции пишут видимость не больше 10 миль, 99 % значений ≤ 10 | обрезаем до 10 |
| `duration_min` | до 5 лет | влияние на движение дольше суток — незакрытое событие | удаляем запись |
| `distance_mi` | 441 миля | длинный хвост — реальные заторы на шоссе | оставляем |

Сначала считаем, сколько записей задевает каждое правило.

In [ ]:
rules = {
    "temperature_f": (F.col("temperature_f") <= -35) | (F.col("temperature_f") > 120),
    "wind_speed_mph": F.col("wind_speed_mph") >= 55,
    "pressure_in": (F.col("pressure_in") < 20) | (F.col("pressure_in") > 32),
    "visibility_mi": F.col("visibility_mi") > 10,
    "duration_min": F.col("duration_min") > 24 * 60,
}

df.select([F.sum(rule.cast("int")).alias(c) for c, rule in rules.items()]).toPandas()

Ошибочных значений погоды — сотни на 7,6 млн записей, видимость больше 10 миль и длительность больше суток — по 0,4 %. Применяем правила.

In [ ]:
df = (
    df
    .withColumn("temperature_f", F.when(rules["temperature_f"], None).otherwise(F.col("temperature_f")))
    .withColumn("wind_speed_mph", F.when(rules["wind_speed_mph"], None).otherwise(F.col("wind_speed_mph")))
    .withColumn("pressure_in", F.when(rules["pressure_in"], None).otherwise(F.col("pressure_in")))
    .withColumn("visibility_mi", F.when(rules["visibility_mi"], 10.0).otherwise(F.col("visibility_mi")))
    .filter(~rules["duration_min"])
)

df.select(list(rules)).summary("min", "max").toPandas().set_index("summary").astype(float).round(2)

Границы теперь правдоподобные. Отдельно проверим самые частые значения длительности.

In [ ]:
(df.groupBy("source", F.round("duration_min", 2).alias("duration_min")).count()
   .orderBy(F.desc("count"))
   .limit(6)
   .toPandas())

Ровно 360 минут у `Source1` и около 30 минут у `Source2` — это значения по умолчанию у источников, а не измеренное время. Длительность — ненадёжный признак: оставляем её в таблице, но в моделях не используем.

<a id="ch6"></a>
## 6. Пропуски

**Путь:** считаем долю пропусков в каждом столбце → для каждого решаем: удалить столбец, заполнить значения или удалить строки.

In [ ]:
null_share = (
    df.select([F.avg(F.col(c).isNull().cast("int")).alias(c) for c in df.columns])
    .toPandas().T[0]
    .sort_values()
)
null_share = null_share[null_share > 0]

null_share.plot.barh(figsize=(8, 5), title="Доля пропусков")
save("missing_share")
plt.show()

null_share.sort_values(ascending=False).map("{:.2%}".format)

Осадки и ощущаемая температура пусты больше чем на четверть. Проверим, когда появились пропуски.

In [ ]:
(df.groupBy(F.year("start_time").alias("year"))
   .agg(F.avg(F.col("precipitation_in").isNull().cast("int")).alias("precipitation_null"),
        F.avg(F.col("wind_chill_f").isNull().cast("int")).alias("wind_chill_null"))
   .orderBy("year")
   .toPandas().round(3))

Пропуски не случайны: до 2019 года оба признака почти не записывали. Заполнить их честно нечем.

* Факт дождя или снега уже есть в `weather_condition` → `precipitation_in` удаляем.
* Ощущаемая температура должна почти повторять обычную. Проверяем корреляцией по строкам, где есть обе величины (агрегат `F.corr` пропуски пропускает; `df.stat.corr` считал бы их нулями).

In [ ]:
df.select(F.corr("temperature_f", "wind_chill_f")).first()[0]

Корреляция 0,99 — `wind_chill_f` дублирует `temperature_f`, удаляем.

In [ ]:
df = df.drop("precipitation_in", "wind_chill_f")

Скорость ветра пуста у 7 % записей — больше, чем другие погодные признаки. Посмотрим, какое направление ветра при этом записано.

In [ ]:
(df.filter(F.col("wind_speed_mph").isNull())
   .groupBy("wind_direction").count()
   .orderBy(F.desc("count"))
   .limit(5)
   .toPandas())

Две трети пропусков — при штиле `Calm`. Но штиль записан и вторым способом, `CALM`. Сравним, что у них со скоростью.

In [ ]:
(df.filter(F.col("wind_direction").isin("Calm", "CALM"))
   .groupBy("wind_direction")
   .agg(F.count("*").alias("records"),
        F.avg(F.col("wind_speed_mph").isNull().cast("int")).alias("speed_null_share"),
        F.max("wind_speed_mph").alias("max_speed"))
   .toPandas())

У `CALM` скорость всегда 0, у `Calm` она не записана ни разу. Это один и тот же штиль в двух форматах, значит пропуск при `Calm` — это ноль.

In [ ]:
df = df.withColumn(
    "wind_speed_mph",
    F.when(F.col("wind_speed_mph").isNull() & (F.col("wind_direction") == "Calm"), 0.0)
     .otherwise(F.col("wind_speed_mph"))
)

Остальные погодные признаки пусты у ~2 % записей. Заполняем их медианой: она устойчива к оставшимся выбросам.

In [ ]:
weather_num = ["temperature_f", "humidity_pct", "pressure_in", "visibility_mi", "wind_speed_mph"]

medians = {c: q[0] for c, q in zip(weather_num, df.approxQuantile(weather_num, [0.5], 0.001))}
medians

In [ ]:
df = df.fillna(medians)

Для словесной погоды и направления ветра заводим явную категорию `Unknown`.

В адресе и времени суток пропусков доли процента — такие строки удаляем.

In [ ]:
df = (
    df
    .fillna({"weather_condition": "Unknown", "wind_direction": "Unknown"})
    .dropna(subset=["street", "city", "timezone", "sunrise_sunset"])
)

Проверяем, что пропусков не осталось.

In [ ]:
remaining = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).first().asDict()
{c: n for c, n in remaining.items() if n > 0}

Пропусков нет.

<a id="ch7"></a>
## 7. Новые признаки

**Путь:** из времени, ветра, погоды и адреса делаем признаки, удобные для анализа и моделей.

**Время.** Год, месяц, день недели (0 — понедельник) и час по местному времени.

In [ ]:
df = (
    df
    .withColumn("year", F.year("start_time"))
    .withColumn("month", F.month("start_time"))
    .withColumn("weekday", F.weekday("start_time"))
    .withColumn("hour", F.hour("start_time"))
)

**Направление ветра.** Смотрим, какие значения встречаются.

In [ ]:
print(sorted(r[0] for r in df.select("wind_direction").distinct().collect()))

Одно и то же направление записано по-разному: `Calm` и `CALM`, `West` и `W`, `Variable` и `VAR`. Сводим к коротким обозначениям.

In [ ]:
wind_names = {"Calm": "CALM", "Variable": "VAR", "North": "N", "South": "S", "East": "E", "West": "W"}
df = df.replace(wind_names, subset=["wind_direction"])

print(sorted(r[0] for r in df.select("wind_direction").distinct().collect()))

**Погода.** В `weather_condition` 144 формулировки: `Light Rain`, `Heavy Rain`, `Rain / Windy` и так далее. Для анализа сводим их в 7 групп по ключевым словам. Порядок правил важен: «дождь с грозой» — это гроза, «дождь со снегом» — снег.

In [ ]:
weather = F.lower("weather_condition")

df = df.withColumn(
    "weather_group",
    F.when(weather.rlike("thunder|t-storm|tornado|funnel"), "storm")
     .when(weather.rlike("snow|sleet|ice|wintry|hail|freezing"), "snow")
     .when(weather.rlike("rain|drizzle|shower"), "rain")
     .when(weather.rlike("fog|mist|haze|smoke|dust|sand|ash"), "fog")
     .when(weather.rlike("cloud|overcast"), "cloudy")
     .when(weather.rlike("fair|clear"), "clear")
     .otherwise("other")
)

df.groupBy("weather_group").count().orderBy(F.desc("count")).toPandas()

**Тип дороги.** Улиц 336 тысяч, но по названию видно класс дороги: межштатная магистраль (`I-95`), шоссе (`US-101`, `CA-91`, `Highway`, `Fwy`) или обычная улица. На магистралях скорости выше — признак может быть связан с тяжестью.

In [ ]:
street = F.col("street")

df = df.withColumn(
    "road_type",
    F.when(street.rlike(r"^I-|\bI-\d|Interstate"), "interstate")
     .when(street.rlike(r"\b(US|SR|Hwy|Highway|Fwy|Freeway|Expy|Expressway|Pkwy|Parkway|Tpke|Turnpike|Route)\b|^[A-Z]{2}-\d"), "highway")
     .otherwise("street")
)

df.select("street", "road_type").sample(0.00002, seed=7).limit(12).toPandas()

Разметка сходится с названиями. Сам `street` больше не нужен.

In [ ]:
df = df.drop("street")
df.groupBy("road_type").count().toPandas()

<a id="ch8"></a>
## 8. Сохранение очищенной таблицы

**Путь:** записываем результат очистки в Iceberg → проверяем объём.

Сохраняем сейчас, до анализа: дальше Spark будет читать из Parquet только нужные столбцы и не повторять цепочку очистки для каждого графика.

In [ ]:
df.writeTo("us_accidents_clean").using("iceberg").createOrReplace()
df.unpersist()

clean = spark.table("us_accidents_clean")
rows_clean = clean.count()

print(f"Было строк:  {rows_raw:,}")
print(f"Стало строк: {rows_clean:,} (удалено {1 - rows_clean / rows_raw:.2%})")
print(f"Столбцов: {len(clean.columns)}")
pd.DataFrame(clean.dtypes, columns=["столбец", "тип"])

<a id="ch9"></a>
## 9. Статистические показатели

**Путь:** считаем среднее, разброс и квартили количественных признаков.

In [ ]:
stat_cols = ["severity", "distance_mi", "duration_min", "temperature_f",
             "humidity_pct", "pressure_in", "visibility_mi", "wind_speed_mph"]

(clean.select(stat_cols)
      .summary()
      .toPandas().set_index("summary").astype(float).T.round(2))

Типичное ДТП: тяжесть 2, около 64 °F (18 °C), влажность 67 %, видимость 10 миль, ветер 7 миль/ч. У протяжённости и длительности среднее заметно больше медианы — длинный правый хвост.

<a id="ch10"></a>
## 10. Распределения

**Путь:** тяжесть → время → место → погода → длительность. Агрегируем в Spark, рисуем маленькие таблицы в pandas.

### 10.1. Тяжесть

In [ ]:
severity = clean.groupBy("severity").count().orderBy("severity").toPandas()
severity["share"] = severity["count"] / severity["count"].sum()

severity.plot.bar(x="severity", y="count", legend=False, figsize=(6, 4), title="Число ДТП по тяжести")
save("severity")
plt.show()
severity.round(3)

Почти 80 % ДТП — тяжесть 2, тяжесть 3–4 — около 20 %, тяжесть 1 — меньше процента.

### 10.2. Время

In [ ]:
monthly = (
    clean.groupBy(F.date_trunc("month", "start_time").alias("month"), "source").count()
    .toPandas()
    .pivot(index="month", columns="source", values="count")
)

monthly.plot(figsize=(10, 4), title="Число ДТП по месяцам и источникам")
save("monthly_by_source")
plt.show()

Число записей определяется охватом источников: до 2020 года основную часть давал `Source2`, затем его вытеснил `Source1`. Летом 2020 года у `Source1` провал почти до нуля — пробел в сборе данных, а не месяц без аварий. Судить по этому ряду о росте аварийности нельзя.

Распределение по дням недели и часам:

In [ ]:
by_hour = (
    clean.groupBy("weekday", "hour").count()
    .toPandas()
    .pivot(index="weekday", columns="hour", values="count")
)
by_hour.index = ["Пн", "Вт", "Ср", "Чт", "Пт", "Сб", "Вс"]

plt.figure(figsize=(12, 4))
sns.heatmap(by_hour, cmap="Reds")
plt.title("Число ДТП по дням недели и часам")
save("weekday_hour")
plt.show()

В будни два пика: 7–8 часов и 15–17 часов — дорога на работу и с работы. В выходные ДТП в 2–3 раза меньше и утреннего пика нет.

### 10.3. Место

In [ ]:
states = clean.groupBy("state").count().orderBy(F.desc("count")).limit(10).toPandas()
states["share"] = states["count"] / rows_clean

states.plot.barh(x="state", y="count", legend=False, figsize=(7, 4), title="10 штатов с наибольшим числом ДТП")
plt.gca().invert_yaxis()
save("top_states")
plt.show()
states.round(3)

Калифорния даёт 23 % всех записей, Флорида — 11 %, Техас — 8 %. Это и население, и разный охват штатов источниками.

Карта: округляем координаты до 0,1° и считаем ДТП в каждой ячейке. Число ДТП в ячейках различается на порядки, поэтому цвет — по логарифму.

In [ ]:
grid = (
    clean.groupBy(F.round("start_lat", 1).alias("lat"), F.round("start_lng", 1).alias("lng"))
    .count()
    .toPandas()
)

plt.figure(figsize=(12, 6))
plt.scatter(grid["lng"], grid["lat"], c=np.log10(grid["count"]), s=1, cmap="inferno_r")
plt.colorbar(label="lg(число ДТП)")
plt.title(f"Очаги ДТП: {len(grid):,} ячеек 0,1° × 0,1°")
plt.xlabel("Долгота")
plt.ylabel("Широта")
save("hotspots_map")
plt.show()

Карта США проступает сама: очаги — крупные города, между ними видна сеть магистралей. Больше всего ДТП на побережьях Калифорнии и Флориды и на северо-востоке.

### 10.4. Погода

In [ ]:
weather_groups = clean.groupBy("weather_group").count().orderBy(F.desc("count")).toPandas()
weather_groups["share"] = weather_groups["count"] / rows_clean
weather_groups.round(3)

85 % ДТП происходят в ясную или облачную погоду — просто потому, что такая погода бывает чаще всего. Влияние погоды поэтому смотрим через долю тяжёлых ДТП (глава 11), а не через их число.

In [ ]:
weather_cols = ["temperature_f", "humidity_pct", "pressure_in", "visibility_mi", "wind_speed_mph"]
sample = clean.select(weather_cols).sample(0.01, seed=42).toPandas()

sample.hist(bins=50, figsize=(14, 6), layout=(2, 3))
plt.suptitle("Распределения погодных признаков (выборка 1 %)")
plt.tight_layout()
save("weather_hist")
plt.show()

Температура распределена почти симметрично, у давления длинный левый хвост (высокогорные станции), видимость почти всегда 10 миль, скорость ветра скошена вправо, а столбец у нуля — штиль. Зубцы — следствие округления при записи, отдельные пики на медианах (влажность 67 %, ветер 7 миль/ч) — заполненные пропуски.

### 10.5. Длительность и протяжённость

In [ ]:
impact = clean.select("source", "duration_min", "distance_mi").sample(0.01, seed=42).toPandas()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(data=impact, x="duration_min", hue="source", bins=100,
             log_scale=(True, False), element="step", ax=axes[0])
axes[0].set_title("Длительность, мин (лог. шкала)")
sns.histplot(data=impact[impact["distance_mi"] > 0], x="distance_mi", hue="source", bins=100,
             log_scale=(True, False), element="step", ax=axes[1])
axes[1].set_title("Протяжённость > 0, мили (лог. шкала)")
plt.tight_layout()
save("duration_distance")
plt.show()

У `Source2` длительность — набор стандартных значений 30, 45, 60 минут, у `Source1` — пики 75 и 360 минут. Протяжённость `Source2` почти всегда 0 или ровно 0,01 мили: этот источник длину участка фактически не измеряет. Оба признака описывают скорее источник, чем само ДТП.

<a id="ch11"></a>
## 11. Тяжесть ДТП и источники данных

**Путь:** проверяем, одинаково ли источники оценивают тяжесть → выбираем данные, на которых можно искать факторы тяжести → ищем их.

In [ ]:
by_source = (
    clean.groupBy("source", "severity").count()
    .toPandas()
    .pivot(index="source", columns="severity", values="count")
)
by_source_share = by_source.div(by_source.sum(axis=1), axis=0)

by_source_share.plot.barh(stacked=True, figsize=(8, 3), title="Доли тяжести внутри источника")
plt.legend(title="severity", bbox_to_anchor=(1, 1))
save("severity_by_source")
plt.show()
by_source_share.round(3)

Источники размечают тяжесть по-разному. Проверим, стабильна ли разметка во времени: доля ДТП с тяжестью 3–4 по годам.

In [ ]:
severe_by_year = (
    clean.groupBy("year", "source")
    .agg(F.avg((F.col("severity") >= 3).cast("int")).alias("severe_share"))
    .toPandas()
    .pivot(index="year", columns="source", values="severe_share")
)

severe_by_year.plot(marker="o", figsize=(8, 4), title="Доля ДТП с тяжестью 3–4")
save("severe_share_by_year")
plt.show()
severe_by_year.round(3)

У `Source1` доля «тяжёлых» упала в десять раз. Проверим, какие уровни тяжести он ставит с 2021 года.

In [ ]:
clean.filter("source = 'Source1' AND year >= 2021").groupBy("severity").count().orderBy("severity").toPandas()

С 2021 года `Source1` не ставит тяжесть 3 вовсе — шкала поменялась. А `Source1` к этому времени даёт большинство записей, поэтому по всему датасету доля тяжёлых ДТП «падает» — это артефакт сбора данных.

У `Source2` доля тяжёлых держится около трети во все годы. Факторы тяжести ищем на нём: признак `severe` — тяжесть 3 или 4.

In [ ]:
s2 = (
    clean.filter(F.col("source") == "Source2")
    .withColumn("severe", (F.col("severity") >= 3).cast("int"))
    .cache()  # дальше по Source2 будет десяток группировок
)
base_share = s2.agg(F.avg("severe")).first()[0]
print(f"Записей Source2: {s2.count():,}, доля тяжёлых: {base_share:.1%}")


def severe_share(column):
    return (s2.groupBy(column)
              .agg(F.avg("severe").alias("severe_share"), F.count("*").alias("count"))
              .orderBy(column)
              .toPandas())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for ax, column in zip(axes.flat[:3], ["road_type", "weather_group", "sunrise_sunset"]):
    data = severe_share(column)
    ax.bar(data[column], data["severe_share"])
    ax.axhline(base_share, color="gray", linestyle="--")
    ax.set_title(f"Доля тяжёлых: {column}")

hours = severe_share("hour")
axes[1, 1].plot(hours["hour"], hours["severe_share"], marker="o")
axes[1, 1].axhline(base_share, color="gray", linestyle="--")
axes[1, 1].set_title("Доля тяжёлых: hour")

plt.tight_layout()
save("severe_factors")
plt.show()

* **Тип дороги** — самый сильный фактор: на межштатных магистралях тяжёлых 80 %, на обычных улицах — 16 %.
* **Погода:** снег, гроза и дождь поднимают долю до 39–44 % против 31 % в ясную погоду.
* **Время:** ночью и после 20 часов тяжёлых около 46 %, в утренний час пик — 25 %: днём много мелких столкновений в пробках.

Объекты рядом с местом ДТП: доля тяжёлых, когда объект есть.

In [ ]:
poi = ["amenity", "bump", "crossing", "give_way", "junction", "no_exit",
       "railway", "roundabout", "station", "stop", "traffic_calming", "traffic_signal"]

poi_share = (
    s2.select([F.avg(F.when(F.col(c), F.col("severe"))).alias(c) for c in poi])
    .toPandas().T[0]
    .sort_values()
)

poi_share.plot.barh(figsize=(7, 5), title="Доля тяжёлых, если объект рядом есть")
plt.axvline(base_share, color="gray", linestyle="--")
save("severe_poi")
plt.show()

Только съезд или развязка (`junction`) повышает долю тяжёлых — до 58 %. Остальные объекты — светофор, переход, знак «стоп» — признаки городской улицы с низкими скоростями, там тяжёлых от 2 до 22 %.

<a id="ch12"></a>
## 12. Корреляции

**Путь:** считаем матрицу корреляций Пирсона количественных признаков одним проходом по данным (`Correlation` из `pyspark.ml`). Берём `Source2`, чтобы тяжесть была сопоставима.

In [ ]:
corr_cols = ["severity", "distance_mi", "duration_min", "temperature_f", "humidity_pct",
             "pressure_in", "visibility_mi", "wind_speed_mph", "start_lat", "start_lng", "hour", "year"]

vectors = VectorAssembler(inputCols=corr_cols, outputCol="features").transform(s2.select(corr_cols))
matrix = Correlation.corr(vectors, "features").first()[0].toArray()
corr = pd.DataFrame(matrix, index=corr_cols, columns=corr_cols)

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Корреляции Пирсона (Source2)")
save("correlation")
plt.show()

Линейные связи слабые, все |r| ≤ 0,42:

* ожидаемые пары: на севере холоднее (широта и температура, −0,41), во влажном воздухе хуже видимость (−0,42), днём теплее и суше (час с температурой +0,22, с влажностью −0,34);
* давление связано с годом (−0,33): с апреля 2019 года вместо давления, приведённого к уровню моря, пишут давление на станции;
* с тяжестью линейно связаны только протяжённость (0,12) и час (0,10).

Тяжесть зависит от признаков нелинейно и через категории (тип дороги, час, погода) — корреляция Пирсона этого не видит, а таблицы долей из главы 11 видят.

<a id="ch13"></a>
## 13. Выводы

**Очистка**

* Удалены 117 695 полных дубликатов (1,5 %), 32 тыс. записей с длительностью больше суток и записи без адреса или времени суток; итог — 7 538 337 записей (−2,5 %).
* Удалены 13 лишних столбцов: константы, копии времени суток, координаты конца участка, текст, служебные и сверхдетальные адреса, а также `precipitation_in` и `wind_chill_f`, которые до 2019 года почти не записывались. Улица `street` заменена типом дороги.
* Ошибочная погода (−89 и 207 °F, ветер 1087 миль/ч, давление 0) переведена в пропуски; видимость обрезана до 10 миль.
* Пропуски ветра при штиле `Calm` заполнены нулём, прочие погодные — медианой, категориальные — значением `Unknown`.
* Добавлены признаки: длительность, год, месяц, день недели, час, группа погоды (7 вместо 144 формулировок), тип дороги; направления ветра сведены к единой записи.

**Анализ**

* Пики ДТП — 7–8 и 15–17 часов в будни; больше всего записей в Калифорнии, Флориде и Техасе.
* Число записей отражает охват источников, а не аварийность: `Source1` вытеснил `Source2`, летом 2020 года — пробел в данных.
* Источники по-разному размечают тяжесть, а `Source1` с 2021 года вообще не ставит тяжесть 3. Анализировать тяжесть можно только внутри источника — на `Source2` она стабильна (34 % тяжёлых).
* Тяжесть сильнее всего зависит от типа дороги (80 % тяжёлых на магистралях против 16 % на улицах), времени суток и погоды; линейные корреляции слабые.

**Что берём в лабораторную № 2**

* Классификация: тяжёлое ли ДТП (тяжесть 3–4) на данных `Source2`; категории — через one-hot кодирование.
* Регрессия: протяжённость затора (`distance_mi`) на данных `Source1` — только он её действительно измеряет.
* Длительность и протяжённость не годятся как признаки: они известны только после ДТП и зависят от источника.

Очищенная таблица: `sobd_accidents.us_accidents_clean`.

In [ ]:
spark.stop()